# LangChain: Q&A over Documents
An example might be a tool that would allow you to query a product catalog for items of interest.

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

llm_model = "gpt-4.1-nano"

In [21]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [22]:
loader = CSVLoader(file_path="assets//OutdoorClothingCatalog_1000.csv")
docs = loader.load()

In [23]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = DocArrayInMemorySearch.from_documents(
    docs,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

In [24]:
llm = ChatOpenAI(
    model=llm_model,  # modern lightweight model
    temperature=0
)

In [25]:
prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question.

Context:
{context}

Question:
{question}
""")

In [26]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [33]:
response = rag_chain.invoke("Please list all shirts with sun protection as markdown list")
print(response.content)

- Sun Shield Shirt by: Provides UPF 50+ sun protection, blocking 98% of harmful UV rays.  
- Men's Tropical Plaid Short-Sleeve Shirt: Rated UPF 50+ for superior UV protection, blocking 98% of harmful rays.  
- Men's Plaid Tropic Shirt, Short-Sleeve: UPF 50+ rated, blocking 98% of UV rays, designed for extended travel and outdoor activities.  
- Tropical Breeze Shirt: Offers UPF 50+ coverage, blocking 98% of harmful UV rays, with SunSmart™ technology for superior sun protection.
